In [1]:
import pandas as pd
import numpy as np

print("=" * 80)
print("Detailed Document Overlap Analysis")
print("=" * 80)

Detailed Document Overlap Analysis


In [2]:
# ============================================================================
# 1. 读取数据
# ============================================================================
print("\n[1] Loading data...")

preference_based = pd.read_csv('result_preference_based_with_text.csv')
preference_df = pd.read_csv('preference.csv')
df_colbert = pd.read_csv('df_colbert_deduped.csv')
df_colbert_prf = pd.read_csv('df_colbert_prf_deduped.csv')

# 类型转换
for df in [preference_based, preference_df, df_colbert, df_colbert_prf]:
    df['qid'] = df['qid'].astype(str)

print(f"  ✓ Loaded all files")

# 尝试读取原始interleaving结果
try:
    original = pd.read_csv('result_interleaved_with_text.csv')
    original['qid'] = original['qid'].astype(str)
    has_original = True
    print(f"  ✓ Original interleaving result available")
except:
    has_original = False
    print(f"  ⚠️ Original interleaving result not found")

preference_map = preference_df.set_index('qid')['preference'].to_dict()


[1] Loading data...
  ✓ Loaded all files
  ✓ Original interleaving result available


In [3]:
# ============================================================================
# 2. 定义分析函数
# ============================================================================

def analyze_overlap_at_k(qids, pref_type, selected_docs_df, k_values=[10, 20, 50, 100]):
    """分析在不同Top-K截断下的重叠度"""
    
    print(f"\n{'=' * 80}")
    print(f"{pref_type} ({len(qids)} queries)")
    print(f"{'=' * 80}")
    
    results = {}
    
    for k in k_values:
        overlap_in_both = 0
        overlap_in_e2e = 0
        overlap_in_prf = 0
        total_docs = 0
        
        rank_stats = []  # 存储排名统计
        
        for qid in qids:
            # 实际选择的文档
            selected = selected_docs_df[selected_docs_df['qid'] == qid]['docno'].tolist()
            
            # E2E和PRF的Top-K
            e2e_topk = set(df_colbert[df_colbert['qid'] == qid]
                          .sort_values('score', ascending=False)
                          .head(k)['docno'])
            prf_topk = set(df_colbert_prf[df_colbert_prf['qid'] == qid]
                          .sort_values('score', ascending=False)
                          .head(k)['docno'])
            
            # E2E和PRF的完整排序（用于查找排名）
            e2e_full = df_colbert[df_colbert['qid'] == qid].sort_values('score', ascending=False)
            prf_full = df_colbert_prf[df_colbert_prf['qid'] == qid].sort_values('score', ascending=False)
            
            e2e_rank_map = {doc: rank+1 for rank, doc in enumerate(e2e_full['docno'])}
            prf_rank_map = {doc: rank+1 for rank, doc in enumerate(prf_full['docno'])}
            
            for doc in selected:
                total_docs += 1
                
                in_e2e_topk = doc in e2e_topk
                in_prf_topk = doc in prf_topk
                
                if in_e2e_topk and in_prf_topk:
                    overlap_in_both += 1
                elif in_e2e_topk:
                    overlap_in_e2e += 1
                elif in_prf_topk:
                    overlap_in_prf += 1
                
                # 记录排名
                e2e_rank = e2e_rank_map.get(doc, None)
                prf_rank = prf_rank_map.get(doc, None)
                
                rank_stats.append({
                    'qid': qid,
                    'docno': doc,
                    'e2e_rank': e2e_rank,
                    'prf_rank': prf_rank,
                    'in_e2e_topk': in_e2e_topk,
                    'in_prf_topk': in_prf_topk
                })
        
        results[k] = {
            'both': overlap_in_both,
            'e2e_only': overlap_in_e2e,
            'prf_only': overlap_in_prf,
            'neither': total_docs - overlap_in_both - overlap_in_e2e - overlap_in_prf,
            'total': total_docs,
            'rank_stats': rank_stats
        }
    
    # 打印结果
    print(f"\nOverlap Analysis at Different Top-K:")
    print(f"{'Top-K':<10} {'Both':>8} {'E2E Only':>10} {'PRF Only':>10} {'Neither':>10} {'Total':>8}")
    print("-" * 70)
    
    for k in k_values:
        r = results[k]
        print(f"Top-{k:<5} {r['both']:>8} ({r['both']/r['total']*100:5.1f}%)  "
              f"{r['e2e_only']:>6} ({r['e2e_only']/r['total']*100:5.1f}%)  "
              f"{r['prf_only']:>6} ({r['prf_only']/r['total']*100:5.1f}%)  "
              f"{r['neither']:>6} ({r['neither']/r['total']*100:5.1f}%)  "
              f"{r['total']:>6}")
    
    # 排名统计
    print(f"\nRanking Statistics (using Top-20 as reference):")
    rank_df = pd.DataFrame(results[20]['rank_stats'])
    
    print(f"  E2E Rankings:")
    print(f"    Mean: {rank_df['e2e_rank'].mean():.1f}")
    print(f"    Median: {rank_df['e2e_rank'].median():.1f}")
    print(f"    Min: {rank_df['e2e_rank'].min():.0f}, Max: {rank_df['e2e_rank'].max():.0f}")
    
    print(f"  PRF Rankings:")
    prf_ranks = rank_df['prf_rank'].dropna()
    if len(prf_ranks) > 0:
        print(f"    Mean: {prf_ranks.mean():.1f}")
        print(f"    Median: {prf_ranks.median():.1f}")
        print(f"    Min: {prf_ranks.min():.0f}, Max: {prf_ranks.max():.0f}")
    
    return results

In [4]:
# ============================================================================
# 3. 分析 PRF-Hurt 查询
# ============================================================================

prf_hurt_qids = preference_df[preference_df['preference'] == 'PRF-Hurt']['qid'].tolist()
if len(prf_hurt_qids) > 0:
    hurt_docs = preference_based[preference_based['qid'].isin(prf_hurt_qids)]
    hurt_results = analyze_overlap_at_k(prf_hurt_qids, "PRF-Hurt (using E2E Top-9)", hurt_docs)
    
    # 如果有原始interleaving，对比
    if has_original:
        print(f"\nComparison with Original Interleaving:")
        orig_hurt = original[original['qid'].isin(prf_hurt_qids)]
        orig_labels = orig_hurt['origin_label'].value_counts()
        
        print(f"  Original interleaving label distribution:")
        for label, count in orig_labels.items():
            print(f"    {label:15s}: {count:3d} ({count/len(orig_hurt)*100:5.1f}%)")
        
        # 计算重叠
        total_overlap = 0
        total_docs = 0
        for qid in prf_hurt_qids:
            new_docs = set(hurt_docs[hurt_docs['qid'] == qid]['docno'])
            orig_docs = set(orig_hurt[orig_hurt['qid'] == qid]['docno'])
            total_overlap += len(new_docs & orig_docs)
            total_docs += len(new_docs)
        
        print(f"\n  Document overlap with original interleaving:")
        print(f"    {total_overlap}/{total_docs} ({total_overlap/total_docs*100:.1f}%) docs are same")


PRF-Hurt (using E2E Top-9) (11 queries)

Overlap Analysis at Different Top-K:
Top-K          Both   E2E Only   PRF Only    Neither    Total
----------------------------------------------------------------------
Top-10          58 ( 58.6%)      41 ( 41.4%)       0 (  0.0%)       0 (  0.0%)      99
Top-20          75 ( 75.8%)      24 ( 24.2%)       0 (  0.0%)       0 (  0.0%)      99
Top-50          90 ( 90.9%)       9 (  9.1%)       0 (  0.0%)       0 (  0.0%)      99
Top-100         93 ( 93.9%)       6 (  6.1%)       0 (  0.0%)       0 (  0.0%)      99

Ranking Statistics (using Top-20 as reference):
  E2E Rankings:
    Mean: 5.0
    Median: 5.0
    Min: 1, Max: 9
  PRF Rankings:
    Mean: 19.5
    Median: 7.0
    Min: 1, Max: 336

Comparison with Original Interleaving:
  Original interleaving label distribution:
    Both           :  42 ( 42.4%)
    A-Only         :  26 ( 26.3%)
    B-Only         :  26 ( 26.3%)
    Easy-Negative  :   5 (  5.1%)

  Document overlap with original inte

In [5]:
# ============================================================================
# 4. 分析 PRF-Benefit 查询
# ============================================================================

prf_benefit_qids = preference_df[preference_df['preference'] == 'PRF-Benefit']['qid'].tolist()
if len(prf_benefit_qids) > 0:
    benefit_docs = preference_based[preference_based['qid'].isin(prf_benefit_qids)]
    benefit_results = analyze_overlap_at_k(prf_benefit_qids, "PRF-Benefit (using PRF Top-9)", benefit_docs)
    
    if has_original:
        print(f"\nComparison with Original Interleaving:")
        orig_benefit = original[original['qid'].isin(prf_benefit_qids)]
        orig_labels = orig_benefit['origin_label'].value_counts()
        
        print(f"  Original interleaving label distribution:")
        for label, count in orig_labels.items():
            print(f"    {label:15s}: {count:3d} ({count/len(orig_benefit)*100:5.1f}%)")
        
        total_overlap = 0
        total_docs = 0
        for qid in prf_benefit_qids:
            new_docs = set(benefit_docs[benefit_docs['qid'] == qid]['docno'])
            orig_docs = set(orig_benefit[orig_benefit['qid'] == qid]['docno'])
            total_overlap += len(new_docs & orig_docs)
            total_docs += len(new_docs)
        
        print(f"\n  Document overlap with original interleaving:")
        print(f"    {total_overlap}/{total_docs} ({total_overlap/total_docs*100:.1f}%) docs are same")


PRF-Benefit (using PRF Top-9) (9 queries)

Overlap Analysis at Different Top-K:
Top-K          Both   E2E Only   PRF Only    Neither    Total
----------------------------------------------------------------------
Top-10          47 ( 58.0%)       0 (  0.0%)      34 ( 42.0%)       0 (  0.0%)      81
Top-20          56 ( 69.1%)       0 (  0.0%)      25 ( 30.9%)       0 (  0.0%)      81
Top-50          65 ( 80.2%)       0 (  0.0%)      16 ( 19.8%)       0 (  0.0%)      81
Top-100         69 ( 85.2%)       0 (  0.0%)      12 ( 14.8%)       0 (  0.0%)      81

Ranking Statistics (using Top-20 as reference):
  E2E Rankings:
    Mean: 39.2
    Median: 7.0
    Min: 1, Max: 427
  PRF Rankings:
    Mean: 5.0
    Median: 5.0
    Min: 1, Max: 9

Comparison with Original Interleaving:
  Original interleaving label distribution:
    Both           :  30 ( 37.0%)
    A-Only         :  23 ( 28.4%)
    B-Only         :  23 ( 28.4%)
    Easy-Negative  :   5 (  6.2%)

  Document overlap with original in

In [7]:
# ============================================================================
# 5. 总结
# ============================================================================

print(f"\n{'=' * 80}")
print("Summary")
print(f"{'=' * 80}")

print(f"""
Key Insights:

1. Top-10 overlap: How many selected docs are in BOTH systems' Top-10?
   - High overlap → The two systems largely agree on top results
   - Low overlap → Selected docs from one system are ranked low by the other

2. Ranking statistics: Where do selected docs rank in the other system?
   - Low mean rank → Other system also thinks these docs are relevant
   - High mean rank → Selected docs are considered less relevant by other system

3. Comparison with original interleaving:
   - Shows what we gained/lost by choosing one system over interleaving
""")

print("\n" + "=" * 80)


Summary

Key Insights:

1. Top-10 overlap: How many selected docs are in BOTH systems' Top-10?
   - High overlap → The two systems largely agree on top results
   - Low overlap → Selected docs from one system are ranked low by the other

2. Ranking statistics: Where do selected docs rank in the other system?
   - Low mean rank → Other system also thinks these docs are relevant
   - High mean rank → Selected docs are considered less relevant by other system

3. Comparison with original interleaving:
   - Shows what we gained/lost by choosing one system over interleaving


